CA02: This is a eMail Spam Classifers that uses Naive Bayes supervised machine learning algorithm. 

In this assignment you will ...
1. Complete the code such a way that it works correctly with this given parts of the program.
2. Explain as clearly as possible what each part of the code is doing. Use "Markdown" texts and code commenting to explain the code

IMPORTANT NOTE:

The path of your data folders 'train-mails' and 'test-mails' must be './train-mails' and './test-mails'. This means you must have your .ipynb file and these folders in the SAME FOLDER in your laptop or Google Drive. The reason for doing this is, this way the peer reviewes and I would be able to run your code from our computers using this exact same relative path, irrespective of our folder hierarchy.

In [11]:
import os
import numpy as np
from collections import Counter

# Import all other necessary libraries. Your code below ...
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

The function, make_Dictionary(), takes in a directory, or folder, of email files, cleans it up, and creates a dictionary of the 3000 most common words.

In [12]:
def make_Dictionary(root_dir):  
    
    all_words = [] #creates a list to hold every word of every email
    
    #Collects file paths and creates a list of them
    emails = [os.path.join(root_dir,f) for f in os.listdir(root_dir)]
    
    #Iterates over every email file and reads them line by line
    for mail in emails:
        with open(mail) as m:
            for line in m:
                words = line.split() #Splits texts after every space. ex) ["I", "like", "bread"]
                all_words += words #Adds words into the list
    
    #Counts the frequency of each word then creates a dictionary of the word and the number of occurences
    dictionary = Counter(all_words) 
    
    #Creates copy of the keys in the dictionary and turns it into a list
    list_to_remove = list(dictionary)
    
    for item in list_to_remove: #for loop that iterates through list and cleans it up
        #deletes non-alphabetical words. ex) "123", "what!!!"
        if item.isalpha() == False: 
            del dictionary[item]
        #deletes single letter words
        elif len(item) == 1: 
            del dictionary[item]
    #Returns a list of tuples that keeps the top 3000 words
    dictionary = dictionary.most_common(3000)
    return dictionary
            

The function, extract_features(), takes in a directory consisting of the email files and then returns two things. 
  - A table, or matrix, where the features are the top 3000 common words and the rows are the email files
  - A list of labels corresponding to each row, where 0 is "not spam" and 1 is "spam"


In [13]:
def extract_features(mail_dir):
    
    #Takes in filepaths and creates a list of them
    files = [os.path.join(mail_dir,fi) for fi in os.listdir(mail_dir)]
    
    #Creates a fixed size table that will store 3000 features.
    #The features are the top 3000 words and each row being an email
    features_matrix = np.zeros((len(files),3000))
    
    #Creates a list of labels stating whether a email is spam or not. 0 being not spam, 1 being spam
    train_labels = np.zeros(len(files)) 
    count = 1; #Tracks many spam emails found
    docID = 0; #tracks the row of the matrix you're filling
    
    #Opens each file and and brings each lines together in a list
    for fil in files:
        with open(fil) as fi:
            for i, line in enumerate(fi):
                if i ==2: #Only processess the words in the 3rd line, ignoring the rest
                    words = line.split() #Splits the words in the line into a list
                    
                    #loop that goes through each word, finds it's index in the 3000 word dictionary, then counts it
                    for word in words:
                        wordID = 0 #the index number that is found
                        for i, d in enumerate(dictionary):
                            if d[0] == word: #checks if the word is in the dictionary
                                wordID = i #if so, it assigns the index number to wordID
                                features_matrix[docID,wordID] = words.count(word) #Sets the word and index into the feature matrix
            
            #This chunk checks labels emails as spam or not
            train_labels[docID] = 0; #not spam by default
            filepathTokens = fil.split('/') #takes a file path and splits each part into a list
            lastToken = filepathTokens[len(filepathTokens)-1] #last part of the file path.(which is the name of the email file)
            
            #if the filename starts with "spmsg", label 1 for spam
            if lastToken.startswith("spmsg"): 
                train_labels[docID] = 1;
                count = count + 1 #counts number of spams
            docID = docID + 1 #moves to next file, then repeats the loop
    return features_matrix, train_labels #returns the table and list of labels to each corresponding row      

In [14]:
#Setting test and train directory for file paths with text files
TRAIN_DIR = '/Users/sarkiss./Desktop/Computer Science/Machine Learning/Naive Baiyes Spam Email /Data/train-mails'
TEST_DIR = "/Users/sarkiss./Desktop/Computer Science/Machine Learning/Naive Baiyes Spam Email /Data/test-mails"

In [15]:
#Creating a dictionary with the training data
dictionary = make_Dictionary(TRAIN_DIR)

#Creating feature able and list of labels for both the training and test data
print ("Reading and processing emails from TRAIN and TEST folders")
features_matrix, labels = extract_features(TRAIN_DIR)
test_features_matrix, test_labels = extract_features(TEST_DIR)
print("Done")

reading and processing emails from TRAIN and TEST folders


Training the Naive Bayes Algorithm, then using the test data to predict if a email is spam or not. Afterwards, we will check the accuracy of the model

In [16]:
print("Reading and processing emails from TRAIN and TEST folders")

print("Training Model using Gaussian Naive Bayes algorithm .....")
model = GaussianNB() #Initializing the Naive Bayes model

model.fit(features_matrix, labels) #Training the model
print("Training Complete")

print("Testing trained model to predict Test Data labels")
predicted_labels = model.predict(test_features_matrix) #Testing the model

print("Completed classification of the Test Data .... now printing Accuracy Score by comparing the Predicted Labels with the Test Labels:")
accuracy = accuracy_score(test_labels, predicted_labels) #Getting model accuracy
print(accuracy)

Reading and processing emails from TRAIN and TEST folders
Training Model using Gaussian Naive Bayes algorithm .....
Training Complete
Testing trained model to predict Test Data labels
Completed classification of the Test Data .... now printing Accuracy Score by comparing the Predicted Labels with the Test Labels:
0.9653846153846154


======================= END OF PROGRAM =========================